In [142]:
import numpy as np
import cvxpy as cp
from scipy.spatial.transform import Rotation
np.set_printoptions(precision=3, suppress=True)

rng = np.random.default_rng()

# Lab-frame reference vectors
m_lab = np.array([0.4, 0.0, 0.9])
m_lab /= np.linalg.norm(m_lab)
a_lab = np.array([0.0, 0.0, 1.0])
T = a_lab @ m_lab

# True sensor distortion: m_raw = D m_true + b
D_true = np.array([
    [1.20,  0.10, -0.05],
    [0.04,  0.85,  0.08],
    [-0.03, 0.06,  1.10],
])
b_true = np.array([0.12, -0.08, 0.05])

 
def sample(noise=0.15):
    R = Rotation.random(random_state=rng).as_matrix()

    a = R @ a_lab + noise * rng.normal(size=3)
    m = D_true @ (R @ m_lab) + b_true + noise * rng.normal(size=3)

    a /= np.linalg.norm(a)
    return m, a

 
# q = [Q11,Q22,Q33,Q44,Q12,Q13,Q14,Q23,Q24,Q34]
def quadratic_features(x):
    return np.array([
        x[0]**2, x[1]**2, x[2]**2, x[3]**2,
        2*x[0]*x[1], 2*x[0]*x[2], 2*x[0]*x[3],
        2*x[1]*x[2], 2*x[1]*x[3], 2*x[2]*x[3],
    ])

 
class StreamingCalibration:
    def __init__(self):
        self.Ha = np.zeros((12, 12))
        self.ga = np.zeros(12)
        self.Hn = np.zeros((10, 10))
        self.gn = np.zeros(10)
        self.n = 0

    def update(self, m_raw, a):
        x = np.r_[m_raw, 1.0]

        # a^T W x = (a kron x)^T vec_row(W)
        z = np.kron(a, x)
        phi = quadratic_features(x)

        self.Ha += np.outer(z, z)
        self.ga += T * z

        self.Hn += np.outer(phi, phi)
        self.gn += phi
        self.n += 1

    def solve(self, trace_weight=1e-4, use_constraint=True, weight=1):
        W = cp.Variable((3, 4))
        q = cp.Variable(10)

        Q = cp.bmat([
            [q[0], q[4], q[5], q[6]],
            [q[4], q[1], q[7], q[8]],
            [q[5], q[7], q[2], q[9]],
            [q[6], q[8], q[9], q[3]],
        ])

        w = cp.reshape(W, (12,), order="C")

        objective = (0
            + (cp.quad_form(w, self.Ha / self.n) - 2 * (self.ga / self.n) @ w ) * weight
            + (cp.quad_form(q, self.Hn / self.n) - 2 * (self.gn / self.n) @ q ) * 1
            + trace_weight * cp.trace(Q)
        )

        constraint = cp.bmat([
            [Q,   W.T],
            [W, np.eye(3)],
        ]) >> 0

        if use_constraint:
            problem = cp.Problem(cp.Minimize(objective), [constraint])
        else:
            problem = cp.Problem(cp.Minimize(objective))
        problem.solve(solver=cp.CLARABEL)

        return W.value, Q.value

 


In [143]:
cal = StreamingCalibration()

for _ in range(100):
    m_raw, a = sample()
    cal.update(m_raw, a)

In [144]:
W_hat, Q_hat = cal.solve(use_constraint=False)
print(W_hat.T @ W_hat - Q_hat)

A_hat = W_hat[:, :3]
b_hat = W_hat[:, 3]

print("Recovered affine map: m_cal = A_hat @ m_raw + b_hat")
print("A_hat:\n", A_hat)
print("b_hat:", b_hat)

# Ground-truth inverse calibration
print("\nTrue inverse:")
print("A_true:\n", np.linalg.inv(D_true))
print("b_true:", -np.linalg.inv(D_true) @ b_true)

# Relaxation tightness
slack = Q_hat - W_hat.T @ W_hat
print("\nSlack eigenvalues:", np.linalg.eigvalsh(slack))
print("Relative slack:",
      np.linalg.norm(slack, "fro") / np.linalg.norm(Q_hat, "fro"))

[[ 0.643 -0.105  0.081 -0.069]
 [-0.105  1.239 -0.15   0.139]
 [ 0.081 -0.15   0.914 -0.057]
 [-0.069  0.139 -0.057 -0.979]]
Recovered affine map: m_cal = A_hat @ m_raw + b_hat
A_hat:
 [[ 0.798 -0.026  0.112]
 [-0.077  1.108 -0.049]
 [-0.013 -0.097  0.948]]
b_hat: [-0.076  0.119 -0.045]

True inverse:
A_true:
 [[ 0.838 -0.102  0.045]
 [-0.042  1.188 -0.088]
 [ 0.025 -0.068  0.915]]
b_true: [-0.111  0.104 -0.054]

Slack eigenvalues: [-1.333 -0.861 -0.615  0.991]
Relative slack: 1.9674847123960675


In [149]:
W_hat, Q_hat = cal.solve(use_constraint=True, weight=5)
print(W_hat.T @ W_hat - Q_hat)

A_hat = W_hat[:, :3]
b_hat = W_hat[:, 3]

print("Recovered affine map: m_cal = A_hat @ m_raw + b_hat")
print("A_hat:\n", A_hat)
print("b_hat:", b_hat)

# Ground-truth inverse calibration
print("\nTrue inverse:")
print("A_true:\n", np.linalg.inv(D_true))
print("b_true:", -np.linalg.inv(D_true) @ b_true)

# Relaxation tightness
slack = Q_hat - W_hat.T @ W_hat
print("\nSlack eigenvalues:", np.linalg.eigvalsh(slack))
print("Relative slack:",
      np.linalg.norm(slack, "fro") / np.linalg.norm(Q_hat, "fro"))

[[-0.001  0.     0.     0.008]
 [ 0.    -0.    -0.    -0.   ]
 [ 0.    -0.    -0.    -0.004]
 [ 0.008 -0.    -0.004 -0.076]]
Recovered affine map: m_cal = A_hat @ m_raw + b_hat
A_hat:
 [[ 0.766 -0.017  0.06 ]
 [-0.075  1.075 -0.074]
 [-0.008 -0.072  0.89 ]]
b_hat: [-0.076  0.105 -0.045]

True inverse:
A_true:
 [[ 0.838 -0.102  0.045]
 [-0.042  1.188 -0.088]
 [ 0.025 -0.068  0.915]]
b_true: [-0.111  0.104 -0.054]

Slack eigenvalues: [0.    0.    0.    0.077]
Relative slack: 0.049010165946833106
